# Object Detection using Transfer Learning of CNN Architectures
**Bird / Cat / Dog localisation with a pre-trained MobileNetV2 backbone**

## Aim
Detect the object in an image (class **and** bounding box) by re-using a CNN pre-trained on a large dataset.

## Steps covered
| Step | Task | Blocks |
|---|---|---|
| a | Load a pre-trained CNN trained on a large dataset (MobileNetV2 / ImageNet) | 6 |
| b | Freeze the weights of the lower convolutional layers | 7 |
| c | Add a custom classifier with several trainable layers (class head + box head) | 8-9 |
| d | Train the new layers on our data | 10-12 |
| e | Fine-tune hyper-parameters and unfreeze more layers | 13-16 |

## Required dataset
One object per image. Images in `dataset_detection/images/` plus a file `dataset_detection/annotations.csv` with these columns (box in **pixels** of the original image):

`filename, width, height, class, xmin, ymin, xmax, ymax`

Example row: `cat_01.jpg, 640, 480, cat, 120, 80, 430, 400`

Labels can be drawn with LabelImg, CVAT or Roboflow and exported to CSV.

**Libraries:** `pip install tensorflow numpy pandas matplotlib scikit-learn pillow`

## Block 1 - Import Libraries
- `pandas`: reads the annotation CSV
- `train_test_split`: splits the data into train / validation / test
- `load_img`, `img_to_array`: read and convert images
- `MobileNetV2` and `preprocess_input`: the pre-trained backbone and its required pixel scaling (-1 to 1)
- `GlobalAveragePooling2D`, `Dense`, `Dropout`: layers for our custom head
- `to_categorical`: converts class numbers to one-hot labels

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

## Block 2 - Set Parameters and Paths
- `IMG_SIZE = 128`: every image is resized to 128 x 128
- `CLASS_NAMES`: order decides the class numbers (bird = 0, cat = 1, dog = 2)
- `EPOCHS_1` / `EPOCHS_2`: epochs for phase 1 (frozen base) and phase 2 (fine-tuning)

**Change `BASE_PATH` to your own dataset folder.**

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS_1 = 10
EPOCHS_2 = 10

BASE_PATH = r"C:\Users\Sairam\OneDrive\Desktop\IT\dataset_detection"
IMAGE_DIR = os.path.join(BASE_PATH, "images")
CSV_PATH = os.path.join(BASE_PATH, "annotations.csv")

CLASS_NAMES = ["bird", "cat", "dog"]
class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
num_classes = len(CLASS_NAMES)

## Block 3 - Read the Annotations
Loads the CSV and shows the first rows and how many images each class has. Check that every class has images before continuing.

In [ ]:
df = pd.read_csv(CSV_PATH)

print(df.head())
print("\nTotal images:", len(df))
print("\nImages per class:")
print(df["class"].value_counts())

## Block 4 - Load Images and Normalise the Boxes
- Each image is resized to 128 x 128.
- Box coordinates are divided by the original width/height so they become values between 0 and 1. This keeps the boxes correct after resizing.
- `X_raw` keeps normal 0-255 images for drawing; `X` is scaled with `preprocess_input` for the network.
- Labels become one-hot vectors.

In [ ]:
images, boxes, labels = [], [], []

for _, row in df.iterrows():
    img = load_img(os.path.join(IMAGE_DIR, row["filename"]), target_size=(IMG_SIZE, IMG_SIZE))
    images.append(img_to_array(img))

    box = [
        row["xmin"] / row["width"],
        row["ymin"] / row["height"],
        row["xmax"] / row["width"],
        row["ymax"] / row["height"],
    ]
    boxes.append(box)
    labels.append(class_to_idx[row["class"]])

X_raw = np.array(images).astype("uint8")
X = preprocess_input(np.array(images).astype("float32"))
y_box = np.clip(np.array(boxes, dtype="float32"), 0.0, 1.0)
y_class = to_categorical(labels, num_classes)

print("Images:", X.shape)
print("Boxes :", y_box.shape)
print("Labels:", y_class.shape)

## Block 5 - Split into Train, Validation and Test
70% train, 15% validation, 15% test. The same indices are used for images, labels and boxes so they stay matched.

*No augmentation is used here because flipping or shifting an image would also require changing its box.*

In [ ]:
idx = np.arange(len(X))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
X_test_raw = X_raw[test_idx]

yc_train, yc_val, yc_test = y_class[train_idx], y_class[val_idx], y_class[test_idx]
yb_train, yb_val, yb_test = y_box[train_idx], y_box[val_idx], y_box[test_idx]

print("Train:", len(X_train), " Validation:", len(X_val), " Test:", len(X_test))

## Block 6 - Step (a): Load the Pre-trained CNN
- `weights='imagenet'`: weights learned on millions of ImageNet images (edges, textures, shapes, fur, feathers...)
- `include_top=False`: removes the original 1000-class layers so we can attach our own head
- `input_shape=(128, 128, 3)`: 128 x 128 colour images

This is the core idea of **transfer learning**: re-use knowledge from a big task for a new, smaller one.

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

## Block 7 - Step (b): Freeze the Pre-trained Layers
Freezing stops the backbone weights from changing. In phase 1 only our new layers should learn; otherwise the random new layers would send large wrong updates back and damage the useful pre-trained features.

In [ ]:
base_model.trainable = False

## Block 8 - Step (c): Add the Custom Head
A shared feature layer feeds **two outputs**:
- `GlobalAveragePooling2D`: turns the feature maps into one vector of 1280 values
- `Dense(256, relu)` + `Dropout(0.5)`: learns to combine features; dropout reduces overfitting
- **Class head** - `Dense(3, softmax)`: probabilities for bird / cat / dog
- **Box head** - `Dense(64, relu)` then `Dense(4, sigmoid)`: predicts `xmin, ymin, xmax, ymax` as values between 0 and 1 (sigmoid keeps them in that range)

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

class_output = Dense(num_classes, activation="softmax", name="class_output")(x)

b = Dense(64, activation="relu")(x)
bbox_output = Dense(4, activation="sigmoid", name="bbox_output")(b)

## Block 9 - Create the Final Model
Joins the backbone input with the two outputs into one Keras `Model`.

In [ ]:
model = Model(inputs=base_model.input, outputs=[class_output, bbox_output])

## Block 10 - Step (d): Compile the Model
Each output has its own loss:
- class head: `categorical_crossentropy` (multi-class classification)
- box head: `mse` (mean squared error between true and predicted box)

`loss_weights` balance the two losses; the box loss is multiplied by 5 because its values are small. The summary should show only the new layers as trainable.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={
        "class_output": "categorical_crossentropy",
        "bbox_output": "mse",
    },
    loss_weights={
        "class_output": 1.0,
        "bbox_output": 5.0,
    },
    metrics={
        "class_output": "accuracy",
        "bbox_output": "mae",
    },
)

model.summary()

## Block 11 - Step (d): Train the Head (Phase 1)
Trains only the new layers while the backbone stays frozen. `history1` keeps the per-epoch results for plotting.

In [ ]:
history1 = model.fit(
    X_train,
    {"class_output": yc_train, "bbox_output": yb_train},
    validation_data=(X_val, {"class_output": yc_val, "bbox_output": yb_val}),
    epochs=EPOCHS_1,
    batch_size=BATCH_SIZE,
)

## Block 12 - Evaluate Before Fine-Tuning
Two measures on the unseen test set:
- **Class accuracy**: percentage of correct classes
- **Mean IoU** (Intersection over Union): how much the predicted box overlaps the true box. 1.0 is a perfect match, 0 means no overlap. A prediction is usually called good when IoU is at least 0.5.

In [ ]:
def compute_iou(a, b):
    x1 = np.maximum(a[:, 0], b[:, 0])
    y1 = np.maximum(a[:, 1], b[:, 1])
    x2 = np.minimum(a[:, 2], b[:, 2])
    y2 = np.minimum(a[:, 3], b[:, 3])

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)

    return inter / (area_a + area_b - inter + 1e-7)


def evaluate_detector(title):
    pred_class, pred_box = model.predict(X_test, verbose=0)
    acc = np.mean(np.argmax(pred_class, axis=1) == np.argmax(yc_test, axis=1))
    ious = compute_iou(pred_box, yb_test)

    print(f"\n{title}")
    print(f"  Class accuracy : {acc:.4f}")
    print(f"  Mean IoU       : {ious.mean():.4f}")
    print(f"  IoU >= 0.5     : {np.mean(ious >= 0.5) * 100:.1f}% of test images")
    return acc, ious.mean()


acc1, iou1 = evaluate_detector("Before fine-tuning")

## Block 13 - Step (e): Unfreeze More Layers
- Early layers learn general features (edges, colours) that help any task, so they stay frozen.
- The last 30 layers learn more task-specific features, so retraining them adapts the model to birds, cats and dogs.
- BatchNormalization layers are kept frozen. With a small dataset, updating their statistics can make training unstable.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Trainable layers in backbone:",
      sum(layer.trainable for layer in base_model.layers))

## Block 14 - Step (e): Recompile with a Smaller Learning Rate
- Compiling must be repeated after changing which layers are trainable, or the change has no effect.
- The learning rate drops from 0.001 to 0.0001 so the pre-trained weights are only adjusted gently.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss={
        "class_output": "categorical_crossentropy",
        "bbox_output": "mse",
    },
    loss_weights={
        "class_output": 1.0,
        "bbox_output": 5.0,
    },
    metrics={
        "class_output": "accuracy",
        "bbox_output": "mae",
    },
)

## Block 15 - Step (e): Fine-Tune the Model (Phase 2)
Trains again with the last 30 backbone layers unfrozen. `history2` is stored separately so both phases can be joined in one graph.

In [ ]:
history2 = model.fit(
    X_train,
    {"class_output": yc_train, "bbox_output": yb_train},
    validation_data=(X_val, {"class_output": yc_val, "bbox_output": yb_val}),
    epochs=EPOCHS_2,
    batch_size=BATCH_SIZE,
)

## Block 16 - Final Evaluation
Compare with Block 12 to see the effect of fine-tuning.

In [ ]:
acc2, iou2 = evaluate_detector("After fine-tuning")

print("\nSummary")
print(f"  Class accuracy : {acc1:.4f} -> {acc2:.4f}")
print(f"  Mean IoU       : {iou1:.4f} -> {iou2:.4f}")

## Block 17 - Plot Training Curves
Three graphs over both phases: total loss, class accuracy and box error (MAE). The dashed line marks where fine-tuning starts. If training improves while validation does not, the model is overfitting.

In [ ]:
def joined(key):
    return history1.history[key] + history2.history[key]

split_line = EPOCHS_1 - 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(joined("loss"), label="Training")
axes[0].plot(joined("val_loss"), label="Validation")
axes[0].set_title("Total Loss")

axes[1].plot(joined("class_output_accuracy"), label="Training")
axes[1].plot(joined("val_class_output_accuracy"), label="Validation")
axes[1].set_title("Class Accuracy")

axes[2].plot(joined("bbox_output_mae"), label="Training")
axes[2].plot(joined("val_bbox_output_mae"), label="Validation")
axes[2].set_title("Box Error (MAE)")

for ax in axes:
    ax.axvline(x=split_line, linestyle="--", color="gray", label="Fine-tuning starts")
    ax.set_xlabel("Epoch")
    ax.legend()

plt.tight_layout()
plt.show()

## Block 18 - Show Detections on Test Images
Draws the **true box (green)** and the **predicted box (red)** on test images, with the predicted class and confidence in the title.

In [ ]:
def show_predictions(n=6):
    n = min(n, len(X_test))
    pred_class, pred_box = model.predict(X_test[:n], verbose=0)

    cols = min(n, 3)
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(5 * cols, 5 * rows))

    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(X_test_raw[i])

        t = yb_test[i] * IMG_SIZE
        ax.add_patch(patches.Rectangle((t[0], t[1]), t[2] - t[0], t[3] - t[1],
                                       fill=False, edgecolor="lime", linewidth=2))

        p = pred_box[i] * IMG_SIZE
        ax.add_patch(patches.Rectangle((p[0], p[1]), p[2] - p[0], p[3] - p[1],
                                       fill=False, edgecolor="red", linewidth=2))

        cls = np.argmax(pred_class[i])
        true_cls = np.argmax(yc_test[i])
        iou = compute_iou(pred_box[i:i + 1], yb_test[i:i + 1])[0]
        ax.set_title(f"Pred: {CLASS_NAMES[cls]} ({pred_class[i][cls]:.2f})\n"
                     f"True: {CLASS_NAMES[true_cls]} | IoU {iou:.2f}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(6)

## Block 19 - Detect on a New Image
Loads any image, predicts the class and box, and draws the box on the **original-size** image. Change `image_path` to your own picture.

In [ ]:
def detect_image(path):
    original = load_img(path)
    w, h = original.size

    small = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = preprocess_input(np.expand_dims(img_to_array(small), axis=0))

    pred_class, pred_box = model.predict(arr, verbose=0)
    cls = np.argmax(pred_class[0])
    conf = pred_class[0][cls]
    box = pred_box[0] * np.array([w, h, w, h])

    plt.figure(figsize=(8, 6))
    plt.imshow(original)
    ax = plt.gca()
    ax.add_patch(patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                   fill=False, edgecolor="red", linewidth=3))
    ax.text(box[0], max(box[1] - 8, 0), f"{CLASS_NAMES[cls]} {conf:.2f}",
            color="white", fontsize=12, backgroundcolor="red")
    plt.axis("off")
    plt.title("Object Detection Result")
    plt.show()


image_path = r"C:\Users\Sairam\OneDrive\Desktop\IT\dataset_detection\test.jpg"
detect_image(image_path)

## Block 20 - Conclusion

In [ ]:
print(f"Transfer learning detector: class accuracy = {acc2:.4f}, mean IoU = {iou2:.4f}")
print("A pre-trained MobileNetV2 backbone was frozen, given a custom class + box head, "
      "trained, and then fine-tuned by unfreezing its last layers.")

## Notes
1. **One object per image:** this model predicts a single box. For several objects in one image use a detector such as YOLO or SSD (see the earlier notebook).
2. **Dataset size:** use at least 50-100 labelled images per class, otherwise accuracy and IoU will be unreliable.
3. **Hyper-parameters to tune (step e):** learning rate, number of unfrozen layers, epochs, dropout rate, `loss_weights` and head size.
4. **Overfitting check:** if validation loss rises while training loss falls, reduce unfrozen layers or increase dropout.
5. Results vary slightly between runs because of random weight initialisation.